# Optimale medewerkerroutes met Google OR-Tools

Beperkingen:
- 20 medewerkers, elk met eigen thuisadres
- 100 cliënten, 5 per medewerker
- Compatibiliteit op basis van honden, katten en rookgedrag
- Cliënten zonder geschikte medewerker worden overgeslagen en genoteerd
- Geen tijdvensters of uurbeperking

# 1. Bibliotheken importeren

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

print('Bibliotheken geladen.')

# 2. Wegennet laden en graph bouwen

In [ ]:
edges_df = pd.read_csv('../output/heerlen_edge_table.csv')
print(f'Aantal edges geladen: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

G = nx.Graph()
node_coords = {}
edge_geom = {}

for _, row in edges_df.iterrows():
    geom = row['geometry']
    coords = list(geom.coords)
    u, v = row['u'], row['v']
    G.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
    node_coords[u] = (coords[0][0], coords[0][1])
    node_coords[v] = (coords[-1][0], coords[-1][1])
    edge_geom[(u, v)] = geom
    edge_geom[(v, u)] = geom

print(f'Graph: {G.number_of_nodes()} knopen, {G.number_of_edges()} takken.')

node_ids = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
kd_tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

def nearest_node(lon, lat):
    _, idx = kd_tree.query([lon, lat])
    return node_ids[idx]

# 3. Medewerkers en cliënten laden

In [ ]:
employee_data = [
    ('employees 1',  50.8872, 5.9812),
    ('employees 2',  50.8895, 5.9820),
    ('employees 3',  50.8883, 5.9830),
    ('employees 4',  50.8855, 5.9795),
    ('employees 5',  50.8945, 5.9660),
    ('employees 6',  50.8878, 5.9808),
    ('employees 7',  50.8948, 5.9700),
    ('employees 8',  50.8870, 5.9825),
    ('employees 9',  50.8868, 5.9817),
    ('employees 10', 50.8785, 5.9750),
    ('employees 11', 50.8840, 5.9810),
    ('employees 12', 50.8860, 5.9835),
    ('employees 13', 50.8850, 5.9880),
    ('employees 14', 50.8890, 5.9822),
    ('employees 15', 50.8710, 5.9920),
    ('employees 16', 50.8810, 5.9680),
    ('employees 17', 50.8952, 5.9672),
    ('employees 18', 50.8875, 5.9805),
    ('employees 19', 50.8940, 5.9665),
    ('employees 20', 50.8790, 5.9760),
]
employees_df = pd.DataFrame(employee_data, columns=['name', 'lat', 'lon'])
employees_df['node'] = employees_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

emp_extra = pd.read_csv('../output/employees.csv')
emp_extra['name'] = emp_extra['name'].str.strip()
employees_df = employees_df.merge(emp_extra[['name', 'dogs', 'cats', 'smokes']], on='name', how='left')
employees_df['dogs'] = employees_df['dogs'].fillna(-1).astype(int)
employees_df['cats'] = employees_df['cats'].fillna(-1).astype(int)
employees_df['smokes'] = employees_df['smokes'].fillna(False).astype(bool)

print(f'Aantal medewerkers: {len(employees_df)}')
print(employees_df[['name','dogs','cats','smokes']].to_string())

In [ ]:
clients_df = pd.read_csv('../output/clients.csv')

# Coordinaten automatisch detecteren
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except:
            pass
coord_col = coord_col or clients_df.columns[0]

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    return (float(parts[0]), float(parts[1])) if len(parts) == 2 else (np.nan, np.nan)

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(lambda x: pd.Series(split_coords(x)))
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index
clients_df['node'] = clients_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

for col in ['dogs', 'cats', 'smokes', 'care_hours']:
    if col not in clients_df.columns:
        clients_df[col] = 0 if col in ['dogs', 'cats'] else (False if col == 'smokes' else 1.0)
clients_df['smokes'] = clients_df['smokes'].astype(bool)

print(f'Aantal cliënten: {len(clients_df)}')

# 4. Reistijdenmatrix berekenen

In [ ]:
N_EMPLOYEES = len(employees_df)
N_CLIENTS = len(clients_df)
N_TOTAL = N_EMPLOYEES + N_CLIENTS
SCALE = 100
CAPACITY = 5

all_nodes = employees_df['node'].tolist() + clients_df['node'].tolist()
unique_sources = list(set(all_nodes))
print(f'Dijkstra uitvoeren vanaf {len(unique_sources)} unieke knopen ...')

dist_from = {}
for i, src in enumerate(unique_sources):
    dist_from[src] = nx.single_source_dijkstra_path_length(G, src, weight='weight')
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(unique_sources)} gereed')

time_matrix = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int64)
for i in range(N_TOTAL):
    src_graph = all_nodes[i]
    lengths = dist_from[src_graph]
    for j in range(N_TOTAL):
        dst_graph = all_nodes[j]
        t = lengths.get(dst_graph, float('inf'))
        time_matrix[i][j] = int(t * SCALE) if t != float('inf') else 10_000_000

print(f'Reistijdmatrix vorm: {time_matrix.shape}')
print(f'Min reistijd: {time_matrix[time_matrix > 0].min() / SCALE:.2f} min')
print(f'Max reistijd: {time_matrix[time_matrix < 10_000_000].max() / SCALE:.2f} min')

# 5. Compatibiliteit berekenen (huisdieren en rook)

In [ ]:
allowed_vehicles_per_client = []
skipped_clients = []

for cid in range(N_CLIENTS):
    client = clients_df.iloc[cid]
    allowed = []
    for vid in range(N_EMPLOYEES):
        emp = employees_df.iloc[vid]
        if emp['dogs'] != -1 and client['dogs'] > emp['dogs']:
            continue
        if emp['cats'] != -1 and client['cats'] > emp['cats']:
            continue
        if not emp['smokes'] and client['smokes']:
            continue
        allowed.append(vid)
    allowed_vehicles_per_client.append(allowed)
    if not allowed:
        skipped_clients.append(cid)

print(f'Cliënten zonder geschikte medewerker: {len(skipped_clients)}')
for cid in skipped_clients:
    c = clients_df.iloc[cid]
    print(f'  ⚠️  Cliënt {cid}: dogs={c["dogs"]}, cats={c["cats"]}, smokes={c["smokes"]} → wordt overgeslagen')

n_allowed = [len(a) for a in allowed_vehicles_per_client]
print(f'\nGemiddeld geschikte medewerkers per cliënt: {np.mean(n_allowed):.1f}')
print(f'Cliënten met slechts 1 optie: {sum(1 for x in n_allowed if x == 1)}')

# 6. Hulpfunctie: routes extraheren

In [ ]:
def extract_routes(solution, routing, manager):
    routes = []
    for vehicle_id in range(N_EMPLOYEES):
        index = routing.Start(vehicle_id)
        nodes = []
        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            nodes.append(node)
            index = solution.Value(routing.NextVar(index))
        nodes.append(manager.IndexToNode(index))
        client_ids = [n - N_EMPLOYEES for n in nodes if n >= N_EMPLOYEES]
        route_time = sum(
            time_matrix[nodes[i]][nodes[i+1]] / SCALE
            for i in range(len(nodes) - 1)
        )
        routes.append({
            'vehicle_id': vehicle_id,
            'nodes': nodes,
            'client_ids': client_ids,
            'route_time': route_time
        })
    return routes

print('extract_routes gedefinieerd. ✅')

# 7. OR-Tools VRP oplossen

In [ ]:
# Cliënten zonder geschikte medewerker worden als optioneel (disjunction) toegevoegd
# zodat OR-Tools ze mag overslaan zonder te crashen.

def create_data_model():
    data = {}
    data['time_matrix'] = time_matrix.tolist()
    data['num_vehicles'] = N_EMPLOYEES
    data['starts'] = list(range(N_EMPLOYEES))
    data['ends'] = list(range(N_EMPLOYEES))
    data['demands'] = [0] * N_EMPLOYEES + [1] * N_CLIENTS
    data['capacities'] = [CAPACITY] * N_EMPLOYEES
    return data

data = create_data_model()
manager = pywrapcp.RoutingIndexManager(
    len(data['time_matrix']),
    data['num_vehicles'],
    data['starts'],
    data['ends']
)
routing = pywrapcp.RoutingModel(manager)

# Reistijd als kost
def time_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node   = manager.IndexToNode(to_index)
    return data['time_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(time_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Capaciteit: max 5 cliënten per medewerker
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 0, data['capacities'], True, 'Capacity'
)

# Compatibiliteit via vehicle constraints
# Cliënten zonder geschikte medewerker worden optioneel gemaakt (disjunction)
solver = routing.solver()
for cid, allowed in enumerate(allowed_vehicles_per_client):
    node_idx = manager.NodeToIndex(N_EMPLOYEES + cid)
    if not allowed:
        # Mag worden overgeslagen — penalty hoger dan elke reistijd
        routing.AddDisjunction([node_idx], 10_000_000)
    else:
        # Sluit niet-toegestane medewerkers uit
        vehicle_var = routing.VehicleVar(node_idx)
        for v in range(N_EMPLOYEES):
            if v not in allowed:
                solver.Add(vehicle_var != v)

# Oplosparameters
# PARALLEL_CHEAPEST_INSERTION werkt beter bij harde enkelvoudige constraints
# dan PATH_CHEAPEST_ARC (die greedy per voertuig bouwt en vastloopt)
search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)
search_params.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_params.time_limit.seconds = 120
search_params.log_search = True

print('VRP wordt opgelost (huisdieren/rook, geen tijdvensters)...')
solution = routing.SolveWithParameters(search_params)

if solution:
    print(f'\n✅ Oplossing gevonden! Totale reistijd: {solution.ObjectiveValue() / SCALE:.1f} min')
    routes = extract_routes(solution, routing, manager)
    # Rapporteer overgeslagen cliënten
    geplande_clients = set(cid for r in routes for cid in r['client_ids'])
    alle_clients = set(range(N_CLIENTS))
    niet_gepland = alle_clients - geplande_clients
    if niet_gepland:
        print(f'\n⚠️  Niet ingepland ({len(niet_gepland)} cliënten):')
        for cid in sorted(niet_gepland):
            c = clients_df.iloc[cid]
            reden = 'geen geschikte medewerker' if cid in skipped_clients else 'capaciteit vol'
            print(f'  Cliënt {cid}: {reden}')
    else:
        print('\n✅ Alle cliënten ingepland.')
else:
    print('\n❌ Geen oplossing gevonden.')
    routes = []

# 8. Samenvatting routes

In [ ]:
if routes:
    print('=== Route samenvatting ===')
    total_minutes = 0
    for r in routes:
        emp = employees_df.loc[r['vehicle_id'], 'name']
        n = len(r['client_ids'])
        t = r['route_time']
        total_minutes += t
        print(f"{emp:15s}: {n} cliënten | {t:5.1f} min reistijd | stops: {r['client_ids']}")
    print(f'\nTotale reistijd alle medewerkers: {total_minutes:.1f} min')
    print(f'Gemiddeld per medewerker: {total_minutes / N_EMPLOYEES:.1f} min')
else:
    print('Geen routes om weer te geven.')

# 9. Dagplanning per medewerker

In [ ]:
def build_schedule(route, employees_df, clients_df):
    nodes = route['nodes']
    client_ids = route['client_ids']
    start_of_day = datetime.strptime('08:00', '%H:%M')

    def min_to_time(minutes):
        return (start_of_day + timedelta(minutes=minutes)).strftime('%H:%M')

    rows = []
    current_time = 0.0

    rows.append({'Stop': 'Thuis (vertrek)', 'Cliënt': '—',
                 'Aankomst': '—', 'Zorgtijd': '—', 'Vertrek': min_to_time(0)})

    for idx, cid in enumerate(client_ids):
        from_node = nodes[idx]
        to_node   = nodes[idx + 1]
        current_time += time_matrix[from_node][to_node] / SCALE
        care_min = clients_df.loc[cid, 'care_hours'] * 60
        departure_time = current_time + care_min
        client_name = clients_df.loc[cid, 'name'] if 'name' in clients_df.columns else f'Cliënt {cid}'
        rows.append({
            'Stop': f'Stop {idx+1}',
            'Cliënt': client_name,
            'Aankomst': min_to_time(current_time),
            'Zorgtijd': f'{int(care_min)} min',
            'Vertrek': min_to_time(departure_time)
        })
        current_time = departure_time

    # Terugreis
    if len(nodes) >= 2 and len(client_ids) > 0:
        current_time += time_matrix[nodes[len(client_ids)]][nodes[-1]] / SCALE

    rows.append({'Stop': 'Thuis (terug)', 'Cliënt': '—',
                 'Aankomst': min_to_time(current_time), 'Zorgtijd': '—', 'Vertrek': '—'})
    return pd.DataFrame(rows)


if routes:
    all_schedules = {}
    for route in routes:
        if route['client_ids']:
            emp_name = employees_df.loc[route['vehicle_id'], 'name']
            df_sched = build_schedule(route, employees_df, clients_df)
            all_schedules[emp_name] = df_sched
            print(f'\n{emp_name}')
            print(df_sched.to_string(index=False))
    print(f'\nPlanning gegenereerd voor {len(all_schedules)} medewerkers.')
else:
    all_schedules = {}
    print('Geen routes beschikbaar.')

# 10. Visualisatie op kaart (Folium)

In [ ]:
if routes:
    EMPLOYEE_COLORS = [
        '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
        '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
        '#aaffc3','#808000','#00bfff','#000075','#808080',
    ]

    def nodes_to_latlon(path_nodes):
        latlon = []
        for i in range(len(path_nodes) - 1):
            u, v = path_nodes[i], path_nodes[i+1]
            geom = edge_geom.get((u, v))
            if geom is None:
                cu, cv = node_coords.get(u), node_coords.get(v)
                if cu: latlon.append((cu[1], cu[0]))
                if cv: latlon.append((cv[1], cv[0]))
                continue
            coords = list(geom.coords)
            cu = node_coords.get(u)
            if cu and len(coords) >= 2:
                if abs(coords[-1][0] - cu[0]) < abs(coords[0][0] - cu[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        return latlon

    def road_segment(node_a, node_b):
        try:
            path = nx.shortest_path(G, source=node_a, target=node_b, weight='weight')
            return nodes_to_latlon(path)
        except nx.NetworkXNoPath:
            ca, cb = node_coords.get(node_a), node_coords.get(node_b)
            result = []
            if ca: result.append((ca[1], ca[0]))
            if cb: result.append((cb[1], cb[0]))
            return result

    center_lat = float(np.mean(node_lats_arr))
    center_lon = float(np.mean(node_lons_arr))
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

    # Achtergrond wegennet
    for _, row in edges_df.iterrows():
        latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
        folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.3).add_to(m)

    # Routes tekenen
    for route in routes:
        vid = route['vehicle_id']
        color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
        emp = employees_df.loc[vid]
        nodes_seq = route['nodes']
        graph_seq = [all_nodes[n] for n in nodes_seq]
        stop_labels = ['thuis'] + [f'stop {k}' for k in range(1, len(route['client_ids'])+1)] + ['thuis']
        for seg_i in range(len(graph_seq) - 1):
            latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i+1])
            if len(latlon) >= 2:
                folium.PolyLine(
                    locations=latlon, color=color, weight=4, opacity=0.85,
                    tooltip=f"{emp['name']} | {stop_labels[seg_i]} → {stop_labels[seg_i+1]}"
                ).add_to(m)

    # Thuisadressen
    for vid, emp in employees_df.iterrows():
        color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
        r = routes[vid] if vid < len(routes) else None
        popup_text = (
            f"{emp['name']}<br>Reistijd: {r['route_time']:.1f} min<br>Cliënten: {r['client_ids']}"
            if r else emp['name']
        )
        folium.Marker(
            location=[emp['lat'], emp['lon']],
            icon=folium.DivIcon(
                html=f'<div style="width:22px;height:22px;background:{color};border:3px solid white;border-radius:50%;box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>',
                icon_size=(22, 22), icon_anchor=(11, 11)
            ),
            popup=folium.Popup(popup_text, max_width=240),
            tooltip=f"{emp['name']} (thuis)"
        ).add_to(m)

    # Cliënten
    client_info = {}
    for route in routes:
        vid = route['vehicle_id']
        color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
        emp_name = employees_df.loc[vid, 'name']
        for stop_num, cid in enumerate(route['client_ids'], start=1):
            client_info[cid] = (color, emp_name, stop_num)

    for _, client in clients_df.iterrows():
        cid = client['client_id']
        if cid not in client_info:
            continue
        color, emp_name, stop_num = client_info[cid]
        client_name = client.get('name', f'Cliënt {cid}')
        folium.CircleMarker(
            location=[client['lat'], client['lon']],
            radius=7, color='white', weight=1.5, fill=True,
            fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(f'<b>{client_name}</b><br>Medewerker: {emp_name}<br>Stop #{stop_num}', max_width=180),
            tooltip=f'Cliënt {cid} | {emp_name} | stop {stop_num}'
        ).add_to(m)

    # Legenda
    legend_rows = ''
    for route in routes:
        vid = route['vehicle_id']
        color = EMPLOYEE_COLORS[vid % len(EMPLOYEE_COLORS)]
        name = employees_df.loc[vid, 'name']
        t = route['route_time']
        cids = route['client_ids']
        legend_rows += f'''
        <tr>
          <td><div style="width:14px;height:14px;background:{color};border:1px solid #ccc;border-radius:3px;margin:2px;"></div></td>
          <td style="padding:1px 8px;white-space:nowrap;"><b>{name}</b></td>
          <td style="color:#444;white-space:nowrap;">{t:.0f} min &nbsp; {cids}</td>
        </tr>'''

    legend_html = f'''
    <div style="position:fixed;bottom:20px;left:20px;z-index:1000;background:rgba(255,255,255,0.96);
                padding:12px 16px;border-radius:8px;font-size:11px;font-family:sans-serif;
                box-shadow:0 2px 12px rgba(0,0,0,.3);max-height:500px;overflow-y:auto;">
      <b style="font-size:13px;">OR-Tools VRP — Routeoverzicht</b><br>
      <span style="color:#777;font-size:10px;">20 medewerkers · 100 cliënten · max 5 stops/medewerker</span>
      <table style="border-collapse:collapse;margin-top:8px;line-height:1.7;">
        <tr><th></th><th style="text-align:left;padding-right:8px;">Medewerker</th><th style="text-align:left;">Reistijd &amp; cliënten</th></tr>
        {legend_rows}
      </table>
    </div>'''
    m.get_root().html.add_child(folium.Element(legend_html))

    # Planningstabel rechtsonder
    if all_schedules:
        schedule_rows = ''
        for emp_name, df_sched in all_schedules.items():
            table_html = df_sched.to_html(index=False, border=0)
            table_html = table_html.replace('<table', '<table style="width:100%;border-collapse:collapse;margin-top:6px;"')
            table_html = table_html.replace('<th>', '<th style="text-align:left;background:#f2f2f2;padding:4px;">')
            table_html = table_html.replace('<td>', '<td style="padding:4px;border-bottom:1px solid #ddd;">')
            schedule_rows += f'''
            <details style="margin-bottom:10px;">
              <summary style="cursor:pointer;font-weight:bold;padding:6px;background:#f9f9f9;border-radius:5px;">{emp_name}</summary>
              <div style="margin-top:8px;overflow-x:auto;">{table_html}</div>
            </details>'''

        schedule_html = f'''
        <div style="position:fixed;bottom:20px;right:20px;z-index:1000;background:rgba(255,255,255,0.96);
                    padding:12px 16px;border-radius:8px;font-size:12px;font-family:sans-serif;
                    box-shadow:0 2px 12px rgba(0,0,0,.3);max-height:450px;overflow-y:auto;width:420px;">
          <b style="font-size:14px;">Dagplanning per medewerker</b><br>
          <span style="color:#777;">klik op een naam om de tabel te zien</span>
          <hr style="margin:8px 0;">
          {schedule_rows}
        </div>'''
        m.get_root().html.add_child(folium.Element(schedule_html))

    output_path = '../output/vrp_routes_map.html'
    m.save(output_path)
    print(f'Kaart opgeslagen: {output_path}')

    from IPython.display import IFrame, display
    display(IFrame(src='vrp_routes_map.html', width='100%', height=700))
else:
    print('Geen routes — kaart wordt niet gegenereerd.')

# 11. Conclusie

Het notebook lost het VRP op met:
- Maximaal 5 cliënten per medewerker
- Compatibiliteit op basis van honden, katten en rookgedrag
- Cliënten zonder geschikte medewerker worden overgeslagen en gerapporteerd
- Minimale totale reistijd

De routes en dagplanningen zijn zichtbaar in de interactieve Folium-kaart.